In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

# -----------------------------
# Dataset (MNIST -> 32x32 input)
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                      download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False,
                                     download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

# -----------------------------
# LeNet for CAM
# -----------------------------
class LeNet_CAM(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet_CAM, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=2)   # -> [16,32,32]
        self.pool1 = nn.AvgPool2d(2, 2)                           # -> [16,16,16]
        self.conv2 = nn.Conv2d(16, 10, kernel_size=3, padding=1)  # -> [10,16,16]
        self.gap = nn.AdaptiveAvgPool2d(1)                        # -> [10,1,1]
        self.fc = nn.Linear(10, num_classes)                      # classifier

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        fmap = x
        x = self.gap(fmap)
        x = x.view(-1, 10)
        out = self.fc(x)
        return out, fmap

# -----------------------------
# Training setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet_CAM().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -----------------------------
# Training loop
# -----------------------------
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs, _ = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Evaluate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs, _ = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(trainloader):.4f}, Test Acc: {acc:.2f}%")

# -----------------------------
# Save model
# -----------------------------
torch.save(model.state_dict(), "lenet_gradcam_16x16.pth")
print("✅ Model saved as lenet_gradcam_16x16.pth")


ModuleNotFoundError: No module named 'torch'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import cv2

# -----------------------------
# Dataset (MNIST -> 32x32 input)
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

testset = torchvision.datasets.MNIST(root='./data', train=False,
                                     download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)

# -----------------------------
# Define same LeNet_CAM model
# -----------------------------
class LeNet_CAM(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet_CAM, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=2)
        self.pool1 = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 10, kernel_size=3, padding=1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(10, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        fmap = x
        x = self.gap(fmap)
        x = x.view(-1, 10)
        out = self.fc(x)
        return out, fmap

# -----------------------------
# Load trained model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet_CAM().to(device)
model.load_state_dict(torch.load("lenet_gradcam_16x16.pth", map_location=device))
model.eval()

# -----------------------------
# Grad-CAM Function
# -----------------------------
def grad_cam(model, image, target_class=None):
    activations, gradients = [], []

    def forward_hook(module, input, output):
        activations.append(output)

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    # Register hooks on the last conv layer
    hook_fwd = model.conv2.register_forward_hook(forward_hook)
    hook_bwd = model.conv2.register_backward_hook(backward_hook)

    # Forward pass
    image = image.to(device)
    output, _ = model(image)
    if target_class is None:
        target_class = output.argmax(dim=1).item()

    # Backward pass
    model.zero_grad()
    loss = output[0, target_class]
    loss.backward()

    # Get stored data
    fmap = activations[0].detach().cpu().numpy()[0]   # [C,H,W]
    grads = gradients[0].detach().cpu().numpy()[0]    # [C,H,W]

    # Compute channel weights
    weights = np.mean(grads, axis=(1, 2))
    cam = np.zeros(fmap.shape[1:], dtype=np.float32)
    for k, w in enumerate(weights):
        cam += w * fmap[k]

    # ReLU and normalize
    cam = np.maximum(cam, 0)
    cam = cam - np.min(cam)
    cam = cam / (cam.max() + 1e-8)
    cam = cv2.resize(cam, (32, 32))

    # Remove hooks
    hook_fwd.remove()
    hook_bwd.remove()
    return cam, target_class

# -----------------------------
# Visualization Function
# -----------------------------
def show_gradcam(image, cam, label, pred):
    img = image.squeeze().cpu().numpy()
    img = (img * 0.5 + 0.5)   # unnormalize
    img = np.uint8(255 * img)

    # Heatmap
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB), 0.5, heatmap, 0.5, 0)

    # Plot
    plt.figure(figsize=(10,3))
    plt.subplot(1,3,1)
    plt.title(f"Input (Label={label})")
    plt.imshow(img, cmap="gray")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.title(f"Grad-CAM (Pred={pred})")
    plt.imshow(cam, cmap="jet")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.title("Overlay")
    plt.imshow(overlay)
    plt.axis("off")
    plt.show()

# -----------------------------
# Run Grad-CAM on few samples
# -----------------------------
for i in range(3):  # visualize 3 random test samples
    images, labels = next(iter(testloader))
    cam, pred_class = grad_cam(model, images, target_class=None)
    show_gradcam(images[0], cam, label=labels.item(), pred=pred_class)


Colab link: https://colab.research.google.com/drive/1_oYQW8Qj87vKhniXAr7gn3pQAsTSQ774#scrollTo=1RoqsUisip5e